<a href="https://colab.research.google.com/github/Shun0212/CodeBERTPretrained/blob/main/BPECMBERTforCloneDetection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# インストール（Colab環境の場合）
!pip install -U transformers>=4.48.0
!pip install datasets
!pip install flash-attn

# Googleドライブのマウント（必要な場合）
from google.colab import drive
drive.mount('/content/drive')

import os
import re
import torch
import random
import numpy as np

from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim

from datasets import load_dataset
from transformers import RobertaTokenizerFast, ModernBertModel, ModernBertConfig, ModernBertForMaskedLM, TrainingArguments, Trainer

# 作業ディレクトリの設定（任意）
os.chdir('/content/drive/MyDrive/CodeModernBERT')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.4/485.4 kB 30.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 19.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.0/6.0 MB 88.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 109.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 86.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 55.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 40.3 MB/s eta 0:00

In [ ]:
from google.colab import userdata
from huggingface_hub import login

# Colab のシークレット環境変数から Hugging Face トークンを取得
huggingface_token = userdata.get("HUGGINGFACE_TOKEN")

if huggingface_token:
    login(huggingface_token)
    print("ログイン成功")
else:
    print("HUGGINGFACE_TOKEN が設定されていません。")


ログイン成功


In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/drive/MyDrive/CodeModernBERT/logs

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from transformers import Trainer, TrainingArguments, RobertaTokenizerFast, ModernBertModel
from sklearn.metrics import accuracy_score, f1_score

# ----- モデル定義（Trainer 用に修正） -----
class CodeCloneDetectionModel(nn.Module):
    def __init__(self, pretrained_model_name):
        super(CodeCloneDetectionModel, self).__init__()
        # 事前学習済みエンコーダ（ModernBERT）をロード
        self.encoder = ModernBertModel.from_pretrained(pretrained_model_name)
        hidden_size = self.encoder.config.hidden_size
        # CLS 埋め込みの絶対差を入力とする分類層
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(hidden_size, 1)  # 出力はスカラー（ロジット）

    def forward(self, input_ids1, attention_mask1, input_ids2, attention_mask2, labels=None):
        # 両方のコード断片をエンコード
        outputs1 = self.encoder(input_ids=input_ids1, attention_mask=attention_mask1)
        outputs2 = self.encoder(input_ids=input_ids2, attention_mask=attention_mask2)
        # CLS トークンの埋め込みを抽出
        cls1 = outputs1.last_hidden_state[:, 0, :]
        cls2 = outputs2.last_hidden_state[:, 0, :]
        # 絶対差を計算し、ドロップアウト後に分類器へ
        diff = torch.abs(cls1 - cls2)
        diff = self.dropout(diff)
        logits = self.classifier(diff).squeeze(-1)  # shape: [batch_size]

        # labels が与えられた場合は損失も計算
        if labels is not None:
            loss_fn = nn.BCEWithLogitsLoss()
            loss = loss_fn(logits, labels.float())
            return {"loss": loss, "logits": logits}
        else:
            return {"logits": logits}

# ----- データセットのロードと前処理（既存のコード） -----
def preprocess_clone_detection(examples, tokenizer, max_length=256):
    enc1 = tokenizer(examples["func1"], padding="max_length", truncation=True, max_length=max_length)
    enc2 = tokenizer(examples["func2"], padding="max_length", truncation=True, max_length=max_length)
    labels = [int(x) for x in examples["label"]]
    return {
        "input_ids1": enc1["input_ids"],
        "attention_mask1": enc1["attention_mask"],
        "input_ids2": enc2["input_ids"],
        "attention_mask2": enc2["attention_mask"],
        "labels": labels
    }

print("データセットをロード中...")
from datasets import load_dataset
ds_train = load_dataset("google/code_x_glue_cc_clone_detection_big_clone_bench", split="train")
ds_val   = load_dataset("google/code_x_glue_cc_clone_detection_big_clone_bench", split="validation")
ds_test  = load_dataset("google/code_x_glue_cc_clone_detection_big_clone_bench", split="test")
print(f"Train: {len(ds_train)}, Validation: {len(ds_val)}, Test: {len(ds_test)}")

# データセットのバランスを確認
train_labels = ds_train["label"]
train_pos = sum([1 for l in train_labels if l == 1])
train_neg = sum([1 for l in train_labels if l == 0])
print(f"Train dataset balance - Positive: {train_pos} ({train_pos/len(train_labels)*100:.2f}%), "
      f"Negative: {train_neg} ({train_neg/len(train_labels)*100:.2f}%)")

# 使用する事前学習済みモデルとトークナイザー
pretrained_model_name = "Shuu12121/CodeHawks-ModernBERT-1.0"
tokenizer = RobertaTokenizerFast.from_pretrained(pretrained_model_name)
print(f"Tokenizer loaded from {pretrained_model_name}, vocab size: {tokenizer.vocab_size}")

# 前処理（マップ処理）
ds_train = ds_train.map(lambda x: preprocess_clone_detection(x, tokenizer, max_length=2048), batched=True)
ds_val   = ds_val.map(lambda x: preprocess_clone_detection(x, tokenizer, max_length=2048), batched=True)
ds_test  = ds_test.map(lambda x: preprocess_clone_detection(x, tokenizer, max_length=2048), batched=True)

# PyTorch テンソル形式に変換
ds_train.set_format(type="torch")
ds_val.set_format(type="torch")
ds_test.set_format(type="torch")

# ----- モデルの初期化 -----
model = CodeCloneDetectionModel(pretrained_model_name)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print("Code Clone Detection Model is ready.")

# ----- 評価指標の定義 -----
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    # BCEWithLogitsLoss を使用しているので、シグモイドを適用して確率に変換
    probs = 1 / (1 + np.exp(-logits))
    preds = (probs >= 0.5).astype(int)
    accuracy = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds)
    return {"accuracy": accuracy, "f1": f1}

# ----- Trainer 用の TrainingArguments の設定 -----
training_args = TrainingArguments(
    output_dir="./saved_models",
    run_name="code_modernbert_mlm_training_v3",
    overwrite_output_dir=True,
    num_train_epochs=1,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=1e-5,
    eval_strategy="steps",   # 各エポック終了時に評価
    logging_steps=100,
    logging_dir='./logs',
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    report_to="tensorboard",
    save_steps=50000,
    eval_steps=50000,
    save_total_limit=2,
    fp16=True,
)

# ----- Trainer のインスタンス作成 -----
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=ds_train,
    eval_dataset=ds_val,
    compute_metrics=compute_metrics,
)

# ----- 学習開始 -----
trainer.train()

# ----- テストデータで最終評価 -----
results = trainer.evaluate(ds_test)
print(results)

# ----- モデル・トークナイザーの保存と Hugging Face Hub へのアップロード -----
model_save_path = "./code_clone_detection_model"

# トークナイザーの保存
tokenizer.save_pretrained(model_save_path)

# モデルの保存 (PyTorchの方式で保存)
torch.save({
    'model_state_dict': model.state_dict(),
    'encoder_config': model.encoder.config,
}, f"{model_save_path}/pytorch_model.bin")

# あるいは、Trainerを使った保存方法
trainer.save_model(model_save_path)

print("モデルとトークナイザーを保存しました。")

データセットをロード中...


Generating train split:   0%|          | 0/901028 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/415416 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/415416 [00:00<?, ? examples/s]

Train: 901028, Validation: 415416, Test: 415416
Train dataset balance - Positive: 450862 (50.04%), Negative: 450166 (49.96%)


...

model.safetensors:   0%|          | 0.00/609M [00:00<?, ?B/s]

You are attempting to use Flash Attention 2.0 with a model not initialized on GPU. Make sure to move the model to GPU after initializing it on CPU with `model.to('cuda')`.


Code Clone Detection Model is ready.


Step,Training Loss,Validation Loss,Accuracy,F1
50000,0.001000,0.077247,0.988501,0.956224


{'eval_loss': 0.08596093207597733, 'eval_accuracy': 0.9882671827758199, 'eval_f1': 0.9570844926566408, 'eval_runtime': 913.5795, 'eval_samples_per_second': 454.712, 'eval_steps_per_second': 28.42, 'epoch': 1.0}
モデルとトークナイザーを保存しました。


In [ ]:
# prompt: 切断する

from google.colab import runtime
runtime.unassign()
